In [1]:
from dotenv import load_dotenv
import os
from pathlib import Path
load_dotenv()
PATH_FINTOC_2022 = Path(os.getenv('PATH_FINTOC_2022'))

In [2]:
FINTOC_EN = PATH_FINTOC_2022/'data'/'en'
FINTOC_EN_PDF =FINTOC_EN/'pdf'
FINTOC_EN_ANNOTS = FINTOC_EN/'annots'
FINTOC_EN_PRED = FINTOC_EN/'preds'

JSON_EXTENSION = "fintoc4.json"

if not FINTOC_EN_PRED.exists():
    FINTOC_EN_PRED.mkdir()
FILE_PATHS = list(FINTOC_EN_PDF.iterdir())[:10]

In [3]:
import pager
from pager.pager_pipe_line import pdf2region_row

from pager.doc_model import MinerPDFModel
from pager.doc_model import LogicTreeModel
from pager.doc_model.converters import PDF2LogicTree

from typing import Dict
import json

pdf_reader = MinerPDFModel({"page_model": pdf2region_row})
pdf2tree = PDF2LogicTree()
tree = LogicTreeModel()

In [4]:
file_path = FILE_PATHS[0]
pdf_reader.read_from_file(file_path)
pdf_reader.extract()
pdf2tree.convert(pdf_reader, tree)


doc_tree = tree.to_dict()

gt_path = FINTOC_EN_ANNOTS/(file_path.name+'.'+JSON_EXTENSION)
with open(gt_path, 'r') as f:
    data = json.load(f)

/Users/macbookair/program/python/PageR/env/lib/python3.13/site-packages/torch_geometric/nn/conv/tag_conv.py:102: UserWarning: Converting sparse tensor to CSR format for more efficient processing. Consider converting your sparse tensor to CSR format beforehand to avoid repeated conversion (got 'torch.sparse_coo')
  return spmm(adj_t, x, reduce=self.aggr)
/Users/macbookair/program/python/PageR/env/lib/python3.13/site-packages/torch_geometric/utils/_spmm.py:71: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:51.)
  src = src.to_sparse_csr()
/Users/macbookair/program/python/PageR/env/lib/python3.13/site-packages/torch/nn/modules/module.py:1776: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include d

In [5]:
from pager import PDF2Img, ImageModel
import matplotlib.pyplot as plt
from ipywidgets import interactive
from ipywidgets.widgets import BoundedIntText

pdf_model = pdf_reader.page_model.page_units[0].sub_model
pdf_model.path=file_path
region_model = pdf_reader.page_model.page_units[2].sub_model
img_model = ImageModel()
pdf2img = PDF2Img()

In [13]:

num_page = 1
def plot_index(num_page):
    plt.figure(dpi=200)
    true_data = [d for d in data if d['page'] == num_page+1]
    for td in true_data:
        print(f"({td['depth']}) {td['text']}")
    plt.figure(dpi=200)
    pdf_model.num_page = num_page
    pdf2img.convert(pdf_model, img_model)
    
    img_model.show()
    page = tree.document.pages[num_page]
    for reg in page.regions:
        if reg.label == 'header':
            reg.segment.plot(text=reg.label, text_size='xx-small', width=0.4)

    for row in page.rows:
        row.segment.plot(width=0.2, color='r')


N = len(tree.document.pages)
interactive(plot_index, num_page=BoundedIntText(
    value=0,
    min=0,
    max=N-1,
    step=1,
    description='Index:',
    disabled=False
))

interactive(children=(BoundedIntText(value=0, description='Index:', max=124), Output(outputs=({'name': 'stdout…